In [ ]:
# installing libraries bitsandbytes & accelerate for enabling efficient quantized (4/8-bit) model loading and smart GPU/CPU placement
!pip install bitsandbytes
!pip install transformers accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Loading in 8-bit quantized mode
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map='auto'
)

In [ ]:
import torch

chat_history = [
    {"role": "system", "content": "You are a chatbot that remembers the user's name and personal details. Greet the user by name if they mention it. Respond directly, and concisely and without elaboration. Do not explain or reason unless explicitly asked."}
]

print("Bot: Hello! I'm your LLM-powered chatbot. Type 'exit' to quit.\n")

while True:
    user_input = input("User: ")
    if user_input.lower() == "exit":
        print("Bot: Goodbye!")
        break

    chat_history.append({"role": "user", "content": user_input})
    chat_history = chat_history[-10:]
    parts = []
    for msg in chat_history:
        if msg["role"] == "system":
            parts.append(msg["content"])
        elif msg["role"] == "user":
            parts.append(f"User: {msg['content']}")
        else:
            parts.append(f"Bot: {msg['content']}")
    prompt = "\n".join(parts) + "\nBot: "


    max_input_tokens = 1024
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_input_tokens)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=500,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    bot_reply = full_output[len(prompt):].strip()


    for stop_token in ['\nUser:', '\nBot:']:
        if stop_token in bot_reply:
            bot_reply = bot_reply.split(stop_token)[0].strip()

    print(f"Bot: {bot_reply}\n")
    chat_history.append({"role": "bot", "content": bot_reply})

    torch.cuda.empty_cache()